# Master notebook: LLM fine-tuning and alignment on a free T4 GPU

**Contents**
- The project: what it is, the components, the practices
- Setup: notebook settings, dependencies, machine check, save folder
- Data pipeline: look, clean, deduplicate, split, check, save
- Later sections are added when we reach them: SFT, ablations, DPO, reward model, PPO, evaluation, serving

---
# The project

## What this project is
We build a **technical Q&A assistant**: it answers technical questions the way a well-voted Stack Exchange answer does.

**Research question:** how far can a small open model be improved for one domain, using instruction tuning, parameter-efficient fine-tuning and preference alignment, on a single free 16 GB T4?

| Choice | Decision |
|---|---|
| Base model | Qwen/Qwen2.5-1.5B-Instruct (fallback: 0.5B if PPO is too slow or unstable) |
| Training method | Parameter-efficient only (LoRA, QLoRA, DoRA, IA3). No full fine-tuning: it does not fit a T4 |
| Data | Stack Exchange paired dataset: a question with a better-voted and a worse-voted answer |
| Hardware | Free T4 (Kaggle or Colab): 16 GB VRAM, no bf16 (so fp16), sessions end, limited disk |

## The components
| # | Component | Question it answers |
|---|---|---|
| Data | Clean, deduplicate, split | Can we trust the numbers we will report? |
| E0 | Baseline of the untrained model | What does the model do before we touch it? |
| E1 / E2 | Instruction tuning with LoRA / QLoRA | Can the model learn the answer style and domain? How much memory does QLoRA save? |
| E3 | Ablations: rank, data size, epochs, DoRA, IA3 | Which settings actually matter? |
| E4 | DPO (preference tuning) | Does learning better vs worse answers help beyond SFT? |
| E5 | Reward model | Can a small model score answer quality? |
| E6 | Miniature PPO / RLHF (custom loop) | What does RLHF look like when we build it ourselves? |
| E7 | DPO vs PPO | Which alignment method works better here? |
| E8 | Final evaluation and error analysis | Where does the final model still fail, and why? |
| Serving | FastAPI + Docker + Gradio demo | Can someone use the models? |

## How the pieces connect
```
raw Stack Exchange data
   -> clean + dedup + split by question ->  sft | pref | val | test
base model -> E0 baseline
           -> SFT (E1/E2/E3) -> DPO (E4) -> compare (E7)
           -> reward model (E5) -> PPO (E6) ->
all models -> same test set -> E8 evaluation -> demo
```

## Practices we follow
| Practice | Why |
|---|---|
| Test set is never used for training or tuning | Otherwise reported scores are fake (Lesson 2) |
| Split by question, not by row | One question has many rows; row splits leak |
| Fixed random seeds | Runs can be repeated and compared |
| One config (YAML) per experiment | Every result can be traced to its settings |
| Checkpoint to persistent storage (Kaggle output, Hugging Face Hub), resume after a disconnect | Sessions can end without warning |
| Every number comes from a real run output | We never invent results; missing numbers stay empty |
| Weights & Biases tracking, adapters and data versioned on the Hugging Face Hub | Experiments stay reproducible |
| Tests and linting on CPU in GitHub Actions; an evaluation gate | A new model that scores worse than the best one is rejected |
| Notebook first, then modules | Learn each step by seeing it work, then move logic into `src/` |

---
# Setup

## Notebook settings on Kaggle
In the notebook settings (right sidebar), set the **Accelerator** to a GPU (T4) and **Internet** to **On**. Internet needs a phone-verified account; without it the dataset cannot be streamed.

## Install dependencies
The data pipeline only needs `datasets`, `transformers` (for the Qwen tokenizer) and `numpy`. Kaggle and Colab usually have them already; this cell installs a package only if it is missing and never upgrades.
**Version pins are pending**: after the machine check below we pin exact versions that match the installed torch/CUDA.
Later parts will add `peft`, `trl`, `bitsandbytes`, `accelerate`, `wandb`.

In [ ]:
!pip install -q datasets transformers numpy

## Check the machine
Kaggle may show two T4s. Our design uses one, so the first lines list the GPUs and then pin the code to the first one. Paste the output back to the mentor.

In [ ]:
!nvidia-smi -L

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # use one T4, as designed

import platform
import torch, datasets, transformers

print("python", platform.python_version())
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpus visible to torch:", torch.cuda.device_count())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("datasets", datasets.__version__)
print("transformers", transformers.__version__)

## Choose where to save files
Kaggle keeps `/kaggle/working` as the notebook's output when you save a version; files elsewhere are lost when the session ends. Later we also push checkpoints to the Hugging Face Hub for safety. Not needed for Step 1, but we set it up once now.

In [ ]:
import os
BASE = "/kaggle/working/llm-alignment-t4"
os.makedirs(BASE, exist_ok=True)
print("working folder:", BASE)

---
# Data pipeline
Planned steps (the list can change as we learn):
1. Look at the raw data
2. Find the site of each question
3. Filter out bad rows
4. Keep one pair per question
5. Remove near-duplicate questions
6. Split into sft / pref / val / test by question
7. Check for leakage
8. Save the files

## Step 1: Look at the raw data

### What
Read a handful of raw rows from the dataset and print them. No cleaning, no splitting.

### Why
You cannot clean data you have never seen. Every later step depends on knowing the exact columns and what they contain.

### How
- The dataset is huge, so we **stream** it: rows are fetched over the network one at a time and nothing big is saved on the notebook's disk.
- `take(5)` stops after 5 rows.
- Each row is a Python dictionary: `qid`, `question`, `date`, `metadata`, `response_j` (the better answer), `response_k` (the worse answer).

### Read 5 rows from the real dataset

In [ ]:
from datasets import load_dataset

stream = load_dataset(
    "lvwerra/stack-exchange-paired",
    data_dir="data/finetune",
    split="train",
    streaming=True,
)
rows = list(stream.take(5))

print("rows read:", len(rows))
print("columns:", list(rows[0].keys()))
print()
for r in rows:
    print(r["qid"], r["metadata"][1], "| chars: question", len(r["question"]),
          "better", len(r["response_j"]), "worse", len(r["response_k"]))

print("\n--- QUESTION ---\n", rows[0]["question"][:300])
print("\n--- BETTER ANSWER (response_j) ---\n", rows[0]["response_j"][:300])
print("\n--- WORSE ANSWER (response_k) ---\n", rows[0]["response_k"][:300])

### Check question
Which column holds the better answer and which the worse one? What in the output shows the rows come from different sites?

**Your answer:** (write it here, then tell the mentor)